# Chapter 1: Why Small Language Models

*Small Language Models in Practice — Haji Gul*

> Why ``small'' is often the right choice; the size/quality trade-off in plain
terms; the local-first stack we will use; and a 10-line program that proves your
environment works.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## The case for going small

A large frontier model is a generalist: it knows a little about everything and
costs a lot to call. Most real products do not need a generalist. They need a
model that is excellent at *one* domain — your support tickets, your legal
clauses, your sensor logs — and that runs where your data already lives.

A **small language model (SLM)** — here, roughly 0.3B to 8B parameters —
gives you four things a giant API model cannot:

[leftmargin=1.4em]
 - **Privacy.** The data never leaves your machine.
 - **Cost.** After download, inference is essentially free.
 - **Latency.** No network round-trip; tokens stream from local memory.
 - **Control.** You pin the weights, the version, and the behavior.

> **The core trade-off.** A smaller model knows less out of the box. We close that gap three ways, each a
chapter in this book: **fine-tuning** (teach it your domain),
**retrieval** (give it your documents at query time), and **agents**
(let it call tools). Quantization and serving then make it cheap to run.

## How small is small enough?

There is no single answer, but a useful rule of thumb for 2026-era models:

@lll@

**Size** & **Runs on** & **Good for** 

0.3–1B & Any laptop CPU & Classification, extraction, simple chat 
1–3B & Laptop GPU / strong CPU & Domain Q&A, RAG, code snippets 
3–8B & 8–16 GB GPU (quantized) & Agents, reasoning, capable assistants 

Start as small as the task allows. You can always scale up; you rarely need to.

## The stack we will use

Everything in this book is open source and runs locally. The setup cell you will
reuse across chapters:

In [ ]:
%%bash
# Core
pip install transformers datasets accelerate
pip install torch        # pick the build matching your CPU/CUDA

# Fine-tuning, RAG, quantization, serving (added per chapter)
pip install peft bitsandbytes
pip install lancedb sentence-transformers
pip install fastapi uvicorn

## Hello, small model

Let us prove the toolchain works. This loads a tiny instruction-tuned model and
generates a reply — on CPU, in well under a minute after the first download.

In [ ]:
from transformers import pipeline

# A small, capable instruct model. Swap for any HF model id you like.
gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",   # uses GPU if present, else CPU
)

messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain what a small language model is in one sentence."},
]

out = gen(messages, max_new_tokens=60, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

> **Tip.** If the download is slow or the model id has changed, any small instruct model
works here, e.g. `HuggingFaceTB/SmolLM2-360M-Instruct` or
`meta-llama/Llama-3.2-1B-Instruct`. The rest of the book is written so
you can substitute models freely.

## Recap and exercise

You learned *when* an SLM beats a giant model, how to pick a size, and you
ran your first local generation.

**Exercise.** Change the user message to a question from your own domain and
compare a 0.5B model with a 1.5B one. Note the difference in answer quality and
in generation speed — that trade-off is the theme of this entire book.